# Semana 9: NumPy y Estadística Descriptiva

## Programación para Ciencia de Datos

### Instituto Politécnico Nacional

---

## Objetivos de la Semana

1. Calcular percentiles, cuartiles y rango intercuartílico (IQR)
2. Detectar valores atípicos (outliers) usando métodos estadísticos
3. Analizar distribuciones de datos con histogramas y medidas de forma
4. Calcular correlaciones entre variables
5. Realizar operaciones matriciales con NumPy
6. Generar datos aleatorios para simulaciones

---

In [ ]:
import numpy as np

# Configuración para reproducibilidad
np.random.seed(42)

# Configuración de impresión
np.set_printoptions(precision=2, suppress=True)

print("NumPy listo para estadística descriptiva")
print(f"Versión: {np.__version__}")

---

## 1. Percentiles y Cuartiles

Los **percentiles** dividen los datos ordenados en 100 partes iguales. El percentil P indica que P% de los datos están por debajo de ese valor.

Los **cuartiles** son percentiles específicos que dividen los datos en 4 partes:

```
DISTRIBUCIÓN DE DATOS Y CUARTILES
═══════════════════════════════════════════════════════════════════════════════

                    25%          25%          25%          25%
            ├──────────────┼──────────────┼──────────────┼──────────────┤
          Mín             Q1            Q2            Q3             Máx
                      (P25)      (Mediana)       (P75)
                              
                       ▼              ▼              ▼
            ┌──────────┬──────────────┬──────────────┬──────────────┐
            │  Primer  │   Segundo    │    Tercer    │   Cuarto     │
            │ Cuartil  │   Cuartil    │   Cuartil    │   Cuartil    │
            │  (25%)   │    (25%)     │    (25%)     │    (25%)     │
            └──────────┴──────────────┴──────────────┴──────────────┘

            ◄─────────────────────────────────────────────────────────►
                              RANGO INTERCUARTÍLICO
                                  IQR = Q3 - Q1
═══════════════════════════════════════════════════════════════════════════════
```

### 1.1 Calculando Percentiles con NumPy

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                         PERCENTILES EN NUMPY
# ═══════════════════════════════════════════════════════════════════

# Datos de ejemplo: Salarios mensuales (en miles de pesos)
salarios = np.array([15, 18, 20, 22, 25, 28, 30, 32, 35, 40, 
                     42, 45, 48, 50, 55, 60, 70, 85, 100, 150])

print("💰 ANÁLISIS DE SALARIOS (miles de pesos)")
print("═" * 50)
print(f"Datos: {salarios}")
print(f"Número de empleados: {len(salarios)}")

# Calcular percentiles específicos
p10 = np.percentile(salarios, 10)   # 10% gana menos que esto
p25 = np.percentile(salarios, 25)   # Q1 - Primer cuartil
p50 = np.percentile(salarios, 50)   # Q2 - Mediana
p75 = np.percentile(salarios, 75)   # Q3 - Tercer cuartil
p90 = np.percentile(salarios, 90)   # 90% gana menos que esto

print(f"\n📊 PERCENTILES")
print(f"   P10 (10%):  ${p10:.1f}k - El 10% gana menos de esto")
print(f"   P25 (Q1):   ${p25:.1f}k - El 25% gana menos de esto")
print(f"   P50 (Med):  ${p50:.1f}k - La mitad gana menos de esto")
print(f"   P75 (Q3):   ${p75:.1f}k - El 75% gana menos de esto")
print(f"   P90 (90%):  ${p90:.1f}k - El 90% gana menos de esto")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    MÚLTIPLES PERCENTILES A LA VEZ
# ═══════════════════════════════════════════════════════════════════

# Calcular varios percentiles en una sola llamada
percentiles_deseados = [10, 25, 50, 75, 90]
valores_percentiles = np.percentile(salarios, percentiles_deseados)

print("📈 PERCENTILES CALCULADOS EN LOTE")
print("─" * 40)
for p, v in zip(percentiles_deseados, valores_percentiles):
    print(f"   P{p:2d}: ${v:6.1f}k")

# También existe np.quantile() que usa fracciones [0, 1]
cuartiles = np.quantile(salarios, [0.25, 0.5, 0.75])
print(f"\n🎯 CUARTILES (usando np.quantile):")
print(f"   Q1 = ${cuartiles[0]:.1f}k")
print(f"   Q2 = ${cuartiles[1]:.1f}k (mediana)")
print(f"   Q3 = ${cuartiles[2]:.1f}k")

### 1.2 Interpretación de Percentiles

Los percentiles son útiles para:
- **Posicionar valores**: ¿En qué percentil está un salario de $45k?
- **Comparar distribuciones**: ¿Cómo se comparan los salarios entre departamentos?
- **Establecer umbrales**: El top 10% de vendedores recibe bonos.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                  ENCONTRAR EL PERCENTIL DE UN VALOR
# ═══════════════════════════════════════════════════════════════════

from scipy import stats  # Para percentileofscore

# ¿En qué percentil está un salario de $45k?
mi_salario = 45

# Método manual: contar cuántos valores son menores
percentil_manual = (np.sum(salarios < mi_salario) / len(salarios)) * 100

print(f"💼 ¿En qué percentil está un salario de ${mi_salario}k?")
print(f"   Percentil aproximado: {percentil_manual:.1f}")
print(f"   Interpretación: Ganas más que el {percentil_manual:.0f}% de los empleados")

# Ejemplo práctico: Identificar el top 10%
umbral_top10 = np.percentile(salarios, 90)
top_10_empleados = salarios[salarios >= umbral_top10]

print(f"\n🏆 TOP 10% DE SALARIOS (>= ${umbral_top10:.1f}k):")
print(f"   {top_10_empleados}")

---

## 2. Rango Intercuartílico (IQR) y Detección de Outliers

El **IQR** mide la dispersión del 50% central de los datos. Es robusto ante valores extremos.

```
MÉTODO IQR PARA DETECCIÓN DE OUTLIERS
═══════════════════════════════════════════════════════════════════════════════

                            IQR = Q3 - Q1

     OUTLIERS          DATOS NORMALES            OUTLIERS
    INFERIORES     ◄────────────────────►       SUPERIORES
        │                                           │
        ▼                                           ▼
   ─────●─────────┬───────────────────────┬─────────●─────────
                  │                       │
            Q1 - 1.5×IQR            Q3 + 1.5×IQR
           (Límite Inferior)       (Límite Superior)

   Cualquier valor FUERA de estos límites se considera OUTLIER

═══════════════════════════════════════════════════════════════════════════════
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    CÁLCULO DEL IQR Y LÍMITES
# ═══════════════════════════════════════════════════════════════════

# Datos con posibles outliers: tiempos de respuesta de servidor (ms)
tiempos = np.array([45, 48, 50, 52, 55, 58, 60, 62, 65, 68,
                    70, 72, 75, 78, 80, 85, 90, 95, 100, 105,
                    5,    # Posible outlier inferior (muy rápido)
                    500,  # Posible outlier superior (muy lento)
                    450]) # Otro outlier superior

print("⏱️  ANÁLISIS DE TIEMPOS DE RESPUESTA (ms)")
print("═" * 55)

# Calcular cuartiles e IQR
Q1 = np.percentile(tiempos, 25)
Q2 = np.percentile(tiempos, 50)  # Mediana
Q3 = np.percentile(tiempos, 75)
IQR = Q3 - Q1

# Calcular límites para outliers
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"\n📊 ESTADÍSTICAS:")
print(f"   Q1 (25%):      {Q1:6.1f} ms")
print(f"   Q2 (mediana):  {Q2:6.1f} ms")
print(f"   Q3 (75%):      {Q3:6.1f} ms")
print(f"   IQR:           {IQR:6.1f} ms")
print(f"\n🚧 LÍMITES PARA OUTLIERS:")
print(f"   Límite inferior: {limite_inferior:6.1f} ms")
print(f"   Límite superior: {limite_superior:6.1f} ms")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                      IDENTIFICAR OUTLIERS
# ═══════════════════════════════════════════════════════════════════

# Crear máscara booleana para outliers
es_outlier = (tiempos < limite_inferior) | (tiempos > limite_superior)

# Separar datos
outliers = tiempos[es_outlier]
datos_normales = tiempos[~es_outlier]

print("🔍 DETECCIÓN DE OUTLIERS")
print("─" * 55)
print(f"   Total de datos:    {len(tiempos)}")
print(f"   Datos normales:    {len(datos_normales)}")
print(f"   Outliers:          {len(outliers)}")

print(f"\n⚠️  OUTLIERS DETECTADOS: {outliers}")

# Clasificar outliers
outliers_inferiores = tiempos[tiempos < limite_inferior]
outliers_superiores = tiempos[tiempos > limite_superior]

if len(outliers_inferiores) > 0:
    print(f"   ↓ Inferiores (muy rápidos): {outliers_inferiores}")
if len(outliers_superiores) > 0:
    print(f"   ↑ Superiores (muy lentos):  {outliers_superiores}")

# Estadísticas con y sin outliers
print(f"\n📈 COMPARACIÓN DE ESTADÍSTICAS:")
print(f"                    Con outliers    Sin outliers")
print(f"   Media:           {np.mean(tiempos):8.1f} ms     {np.mean(datos_normales):8.1f} ms")
print(f"   Desv. Estándar:  {np.std(tiempos):8.1f} ms     {np.std(datos_normales):8.1f} ms")

### 2.1 Función Reutilizable para Detectar Outliers

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                 FUNCIÓN PARA DETECCIÓN DE OUTLIERS
# ═══════════════════════════════════════════════════════════════════

def detectar_outliers_iqr(datos, factor=1.5):
    """
    Detecta outliers usando el método IQR.
    
    Parámetros:
    -----------
    datos : array-like
        Datos a analizar
    factor : float
        Multiplicador del IQR (1.5 estándar, 3.0 para extremos)
    
    Retorna:
    --------
    dict con: Q1, Q3, IQR, límites, máscara de outliers, outliers
    """
    datos = np.asarray(datos)
    Q1 = np.percentile(datos, 25)
    Q3 = np.percentile(datos, 75)
    IQR = Q3 - Q1
    
    limite_inf = Q1 - factor * IQR
    limite_sup = Q3 + factor * IQR
    
    mascara = (datos < limite_inf) | (datos > limite_sup)
    
    return {
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'limite_inferior': limite_inf,
        'limite_superior': limite_sup,
        'es_outlier': mascara,
        'outliers': datos[mascara],
        'n_outliers': np.sum(mascara)
    }

# Probar la función
resultado = detectar_outliers_iqr(tiempos)
print("🔧 FUNCIÓN detectar_outliers_iqr()")
print("─" * 40)
for clave, valor in resultado.items():
    if clave != 'es_outlier':
        print(f"   {clave}: {valor}")

---

## 3. Detección de Outliers con Z-Score

El **Z-Score** mide cuántas desviaciones estándar está un valor de la media.

```
MÉTODO Z-SCORE PARA DETECCIÓN DE OUTLIERS
═══════════════════════════════════════════════════════════════════════════════

                        x - μ
               Z  =  ─────────
                         σ

   Donde:
   • x = valor individual
   • μ = media de los datos
   • σ = desviación estándar

   INTERPRETACIÓN:
   ───────────────────────────────────────────────────────────────────
   │ Z-Score │ Interpretación                    │ % datos (normal) │
   ├─────────┼───────────────────────────────────┼──────────────────┤
   │ |Z| < 1 │ Dentro de 1 desv. est.           │     68.3%        │
   │ |Z| < 2 │ Dentro de 2 desv. est.           │     95.4%        │
   │ |Z| < 3 │ Dentro de 3 desv. est.           │     99.7%        │
   │ |Z| > 3 │ OUTLIER (muy raro)               │     0.3%         │
   ───────────────────────────────────────────────────────────────────

═══════════════════════════════════════════════════════════════════════════════
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                     CÁLCULO DE Z-SCORE
# ═══════════════════════════════════════════════════════════════════

# Datos de ejemplo: Puntajes de un examen
puntajes = np.array([65, 70, 72, 75, 78, 80, 82, 85, 88, 90,
                     92, 95, 98, 100, 102, 105,
                     25,   # Posible outlier inferior
                     150]) # Posible outlier superior

# Calcular Z-Score para cada valor
media = np.mean(puntajes)
std = np.std(puntajes)
z_scores = (puntajes - media) / std

print("📝 ANÁLISIS DE PUNTAJES CON Z-SCORE")
print("═" * 55)
print(f"Media: {media:.2f}")
print(f"Desviación estándar: {std:.2f}")

print(f"\n{'Puntaje':<10} {'Z-Score':<10} {'Interpretación':<20}")
print("─" * 45)

for puntaje, z in zip(puntajes, z_scores):
    if abs(z) > 3:
        interpretacion = "⚠️  OUTLIER"
    elif abs(z) > 2:
        interpretacion = "🔶 Inusual"
    else:
        interpretacion = "✅ Normal"
    print(f"{puntaje:<10} {z:<10.2f} {interpretacion}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                FUNCIÓN PARA DETECCIÓN CON Z-SCORE
# ═══════════════════════════════════════════════════════════════════

def detectar_outliers_zscore(datos, umbral=3):
    """
    Detecta outliers usando el método Z-Score.
    
    Parámetros:
    -----------
    datos : array-like
        Datos a analizar
    umbral : float
        Z-Score límite (típicamente 2 o 3)
    
    Retorna:
    --------
    dict con: media, std, z_scores, máscara de outliers, outliers
    """
    datos = np.asarray(datos)
    media = np.mean(datos)
    std = np.std(datos)
    z_scores = (datos - media) / std
    
    mascara = np.abs(z_scores) > umbral
    
    return {
        'media': media,
        'std': std,
        'z_scores': z_scores,
        'es_outlier': mascara,
        'outliers': datos[mascara],
        'n_outliers': np.sum(mascara)
    }

# Comparar ambos métodos
resultado_iqr = detectar_outliers_iqr(puntajes)
resultado_zscore = detectar_outliers_zscore(puntajes, umbral=3)

print("🔄 COMPARACIÓN DE MÉTODOS")
print("═" * 50)
print(f"\nMétodo IQR (factor=1.5):")
print(f"   Outliers encontrados: {resultado_iqr['outliers']}")

print(f"\nMétodo Z-Score (umbral=3):")
print(f"   Outliers encontrados: {resultado_zscore['outliers']}")

---

## 4. Distribución de Datos

Entender la **forma** de la distribución es crucial para elegir métodos estadísticos apropiados.

```
FORMAS COMUNES DE DISTRIBUCIONES
═══════════════════════════════════════════════════════════════════════════════

  SIMÉTRICA (Normal)           SESGADA DERECHA          SESGADA IZQUIERDA
  (Skewness ≈ 0)              (Skewness > 0)           (Skewness < 0)

        ▲                           ▲                          ▲
       ╱│╲                         ╱│                          │╲
      ╱ │ ╲                       ╱ │                          │ ╲
     ╱  │  ╲                     ╱  │╲                        ╱│  ╲
    ╱   │   ╲                   ╱   │ ╲___                ___╱ │   ╲
   ╱    │    ╲                 ╱    │     ╲___        ___╱     │    ╲
  ─────────────────       ─────────────────────    ─────────────────────
       media                    media →                  ← media
       mediana                  mediana                    mediana

  Ejemplo: Estaturas         Ejemplo: Ingresos       Ejemplo: Edades de
  de adultos                 salariales              jubilación

═══════════════════════════════════════════════════════════════════════════════
```

### 4.1 Histogramas con NumPy

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    HISTOGRAMAS CON NUMPY
# ═══════════════════════════════════════════════════════════════════

# Generar datos de ejemplo: Edades de clientes
np.random.seed(42)
edades = np.concatenate([
    np.random.normal(35, 10, 500),   # Grupo principal
    np.random.normal(55, 5, 200)     # Grupo secundario
])
edades = np.clip(edades, 18, 80)  # Limitar rango realista

# Crear histograma con numpy
frecuencias, bordes = np.histogram(edades, bins=10)

print("👥 DISTRIBUCIÓN DE EDADES DE CLIENTES")
print("═" * 60)

# Visualización ASCII del histograma
max_freq = max(frecuencias)
escala = 40 / max_freq

print(f"\n{'Rango':<15} {'Frecuencia':<12} {'Histograma'}")
print("─" * 60)

for i in range(len(frecuencias)):
    rango = f"{bordes[i]:.0f}-{bordes[i+1]:.0f}"
    barra = "█" * int(frecuencias[i] * escala)
    print(f"{rango:<15} {frecuencias[i]:<12} {barra}")

print(f"\n📊 Total de clientes: {len(edades)}")

### 4.2 Medidas de Forma: Asimetría (Skewness) y Curtosis

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    ASIMETRÍA Y CURTOSIS
# ═══════════════════════════════════════════════════════════════════

def calcular_skewness(datos):
    """Calcula la asimetría (skewness) de los datos."""
    n = len(datos)
    media = np.mean(datos)
    std = np.std(datos, ddof=0)
    return np.mean(((datos - media) / std) ** 3)

def calcular_kurtosis(datos):
    """Calcula la curtosis de los datos (exceso sobre normal)."""
    n = len(datos)
    media = np.mean(datos)
    std = np.std(datos, ddof=0)
    return np.mean(((datos - media) / std) ** 4) - 3

# Generar diferentes distribuciones
np.random.seed(42)
normal = np.random.normal(50, 10, 1000)
sesgada_derecha = np.random.exponential(10, 1000) + 20
sesgada_izquierda = 100 - np.random.exponential(10, 1000)

print("📐 MEDIDAS DE FORMA")
print("═" * 60)
print(f"\n{'Distribución':<25} {'Skewness':<12} {'Curtosis':<12} {'Interpretación'}")
print("─" * 70)

distribuciones = [
    ("Normal", normal),
    ("Sesgada derecha (+)", sesgada_derecha),
    ("Sesgada izquierda (-)", sesgada_izquierda)
]

for nombre, datos in distribuciones:
    skew = calcular_skewness(datos)
    kurt = calcular_kurtosis(datos)
    
    if abs(skew) < 0.5:
        interp = "Simétrica"
    elif skew > 0:
        interp = "Cola larga a la derecha"
    else:
        interp = "Cola larga a la izquierda"
    
    print(f"{nombre:<25} {skew:<12.3f} {kurt:<12.3f} {interp}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#          INTERPRETACIÓN DE SKEWNESS Y KURTOSIS
# ═══════════════════════════════════════════════════════════════════

print("""
📚 GUÍA DE INTERPRETACIÓN
═══════════════════════════════════════════════════════════════════

SKEWNESS (Asimetría):
─────────────────────────────────────────────────────────────────
  • Skew ≈ 0      : Distribución simétrica
  • Skew > 0      : Sesgada a la derecha (cola hacia valores altos)
  • Skew < 0      : Sesgada a la izquierda (cola hacia valores bajos)
  • |Skew| < 0.5  : Aproximadamente simétrica
  • |Skew| > 1    : Fuertemente sesgada

CURTOSIS (Exceso):
─────────────────────────────────────────────────────────────────
  • Kurt ≈ 0      : Similar a distribución normal (mesocúrtica)
  • Kurt > 0      : Colas pesadas, pico agudo (leptocúrtica)
  • Kurt < 0      : Colas ligeras, pico plano (platicúrtica)

IMPLICACIONES PRÁCTICAS:
─────────────────────────────────────────────────────────────────
  • Datos sesgados: La media no es representativa, usar mediana
  • Curtosis alta: Más outliers de lo esperado
  • Para ML: Considerar transformaciones (log, sqrt)
═══════════════════════════════════════════════════════════════════
""")

---

## 5. Correlación y Covarianza

La **correlación** mide la relación lineal entre dos variables.

```
INTERPRETACIÓN DE CORRELACIÓN
═══════════════════════════════════════════════════════════════════════════════

    r = -1          r = 0           r = +1
    ────────────────────────────────────────
    Perfecta        Sin             Perfecta
    Negativa        Correlación     Positiva

    ●                  ●  ●              ●
     ●               ●    ●             ●
      ●            ●  ●  ●  ●          ●
       ●          ●    ●    ●         ●
        ●        ●   ●   ●   ●       ●
         ●      ●  ●  ●  ●  ● ●     ●

   MAGNITUD:
   ─────────────────────────────────────────
   │ |r|     │ Fuerza de la correlación     │
   ├─────────┼──────────────────────────────┤
   │ 0.0-0.3 │ Débil o ninguna              │
   │ 0.3-0.5 │ Moderada                     │
   │ 0.5-0.7 │ Fuerte                       │
   │ 0.7-1.0 │ Muy fuerte                   │
   ─────────────────────────────────────────

═══════════════════════════════════════════════════════════════════════════════
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                     CORRELACIÓN CON NUMPY
# ═══════════════════════════════════════════════════════════════════

# Datos de ejemplo: Variables de rendimiento de estudiantes
np.random.seed(42)
n_estudiantes = 100

# Horas de estudio
horas_estudio = np.random.uniform(1, 10, n_estudiantes)

# Calificación (correlacionada positivamente con horas)
calificacion = 50 + 5 * horas_estudio + np.random.normal(0, 8, n_estudiantes)
calificacion = np.clip(calificacion, 0, 100)

# Horas de videojuegos (correlacionada negativamente)
horas_videojuegos = 12 - horas_estudio + np.random.normal(0, 2, n_estudiantes)
horas_videojuegos = np.clip(horas_videojuegos, 0, 15)

# Horas de sueño (no correlacionada)
horas_sueño = np.random.uniform(5, 9, n_estudiantes)

# Calcular matriz de correlación
# np.corrcoef espera variables en filas
datos = np.array([horas_estudio, calificacion, horas_videojuegos, horas_sueño])
matriz_corr = np.corrcoef(datos)

variables = ['Hrs Estudio', 'Calificación', 'Hrs Videojuegos', 'Hrs Sueño']

print("📊 MATRIZ DE CORRELACIÓN")
print("═" * 70)
print(f"\n{'':>15}", end='')
for v in variables:
    print(f"{v:>15}", end='')
print()
print("─" * 75)

for i, var in enumerate(variables):
    print(f"{var:>15}", end='')
    for j in range(len(variables)):
        valor = matriz_corr[i, j]
        # Colorear según fuerza
        if i == j:
            print(f"{'1.00':>15}", end='')
        elif abs(valor) > 0.7:
            print(f"{valor:>15.2f}*", end='')  # Fuerte
        elif abs(valor) > 0.3:
            print(f"{valor:>15.2f}", end='')   # Moderada
        else:
            print(f"{valor:>15.2f}", end='')   # Débil
    print()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                     COVARIANZA VS CORRELACIÓN
# ═══════════════════════════════════════════════════════════════════

# Calcular covarianza
matriz_cov = np.cov(datos)

print("📐 COVARIANZA VS CORRELACIÓN")
print("═" * 60)

print("\n📈 COVARIANZA: Mide la variación conjunta (afectada por escalas)")
print(f"   Cov(Estudio, Calificación) = {matriz_cov[0,1]:.2f}")
print(f"   Cov(Estudio, Videojuegos)  = {matriz_cov[0,2]:.2f}")

print("\n📊 CORRELACIÓN: Normalizada entre -1 y 1 (independiente de escalas)")
print(f"   Corr(Estudio, Calificación) = {matriz_corr[0,1]:.2f}")
print(f"   Corr(Estudio, Videojuegos)  = {matriz_corr[0,2]:.2f}")

print("""
💡 DIFERENCIA CLAVE:
   • Covarianza: Puede ser cualquier número
   • Correlación: Siempre entre -1 y 1
   
   La correlación es más útil para comparar relaciones
   entre diferentes pares de variables.
""")

---

## 6. Operaciones Matriciales

NumPy permite realizar operaciones de álgebra lineal eficientemente.

```
OPERACIONES MATRICIALES COMUNES
═══════════════════════════════════════════════════════════════════════════════

TRANSPOSICIÓN (A.T):
┌─────────┐         ┌───────┐
│ 1  2  3 │         │ 1  4  │
│ 4  5  6 │   →     │ 2  5  │
└─────────┘         │ 3  6  │
  (2×3)             └───────┘
                      (3×2)

PRODUCTO PUNTO (np.dot):
┌───────┐   ┌───┐     ┌────┐
│ 1  2  │   │ 5 │     │ 19 │   (1×5 + 2×7 = 19)
│ 3  4  │ × │ 7 │  =  │ 43 │   (3×5 + 4×7 = 43)
└───────┘   └───┘     └────┘
 (2×2)      (2×1)      (2×1)

═══════════════════════════════════════════════════════════════════════════════
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    OPERACIONES MATRICIALES
# ═══════════════════════════════════════════════════════════════════

# Crear matrices de ejemplo
A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 10],
              [11, 12]])

print("🔢 OPERACIONES MATRICIALES")
print("═" * 50)

print(f"\nMatriz A (shape {A.shape}):")
print(A)

print(f"\nMatriz B (shape {B.shape}):")
print(B)

# Transposición
print(f"\n📐 TRANSPOSICIÓN")
print(f"A.T (shape {A.T.shape}):")
print(A.T)

# Producto punto (multiplicación matricial)
print(f"\n✖️  PRODUCTO PUNTO")
print(f"A @ B (o np.dot(A, B)):")
C = np.dot(A, B)  # Equivalente a A @ B
print(f"Resultado (shape {C.shape}):")
print(C)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                 OPERACIONES SOBRE EJES (AXIS)
# ═══════════════════════════════════════════════════════════════════

# Datos: Ventas por tienda (filas) y mes (columnas)
ventas = np.array([
    [100, 120, 130, 140],  # Tienda Norte
    [90, 110, 100, 120],   # Tienda Sur
    [150, 160, 170, 180]   # Tienda Centro
])

tiendas = ['Norte', 'Sur', 'Centro']
meses = ['Ene', 'Feb', 'Mar', 'Abr']

print("🏪 VENTAS POR TIENDA Y MES")
print("═" * 50)
print(f"\n{'Tienda':<10}", end='')
for mes in meses:
    print(f"{mes:>8}", end='')
print(f"{'TOTAL':>10}")
print("─" * 50)

for i, tienda in enumerate(tiendas):
    print(f"{tienda:<10}", end='')
    for j in range(len(meses)):
        print(f"{ventas[i,j]:>8}", end='')
    print(f"{np.sum(ventas[i]):>10}")

# Operaciones sobre ejes
print(f"\n📊 OPERACIONES SOBRE EJES:")
print(f"\n   axis=0 (sobre filas → resultado por columna/mes):")
print(f"   Suma por mes:     {np.sum(ventas, axis=0)}")
print(f"   Promedio por mes: {np.mean(ventas, axis=0)}")

print(f"\n   axis=1 (sobre columnas → resultado por fila/tienda):")
print(f"   Suma por tienda:     {np.sum(ventas, axis=1)}")
print(f"   Promedio por tienda: {np.mean(ventas, axis=1)}")

print(f"\n   Sin axis (sobre todo el array):")
print(f"   Suma total:     {np.sum(ventas)}")
print(f"   Promedio total: {np.mean(ventas):.1f}")

---

## 7. Generación de Datos Aleatorios

NumPy proporciona herramientas para generar datos aleatorios útiles en simulaciones y pruebas.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               GENERACIÓN DE DATOS ALEATORIOS
# ═══════════════════════════════════════════════════════════════════

# IMPORTANTE: Fijar semilla para reproducibilidad
np.random.seed(42)

print("🎲 GENERACIÓN DE DATOS ALEATORIOS")
print("═" * 60)

# 1. Distribución Uniforme
uniforme = np.random.uniform(low=0, high=100, size=5)
print(f"\n1️⃣  UNIFORME (0 a 100):")
print(f"   {uniforme.round(2)}")

# 2. Distribución Normal (Gaussiana)
normal = np.random.normal(loc=50, scale=10, size=5)  # media=50, std=10
print(f"\n2️⃣  NORMAL (μ=50, σ=10):")
print(f"   {normal.round(2)}")

# 3. Enteros aleatorios
enteros = np.random.randint(low=1, high=7, size=10)  # Simular dados
print(f"\n3️⃣  ENTEROS (dados 1-6):")
print(f"   {enteros}")

# 4. Muestreo de un array
opciones = np.array(['A', 'B', 'C', 'D'])
muestra = np.random.choice(opciones, size=6, replace=True)
print(f"\n4️⃣  MUESTREO (con reemplazo):")
print(f"   Opciones: {opciones}")
print(f"   Muestra:  {muestra}")

# 5. Muestreo ponderado
probabilidades = [0.5, 0.3, 0.15, 0.05]  # A más probable, D menos
muestra_pond = np.random.choice(opciones, size=10, p=probabilidades)
print(f"\n5️⃣  MUESTREO PONDERADO:")
print(f"   Probabilidades: {probabilidades}")
print(f"   Muestra: {muestra_pond}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#              EJEMPLO PRÁCTICO: SIMULACIÓN DE VENTAS
# ═══════════════════════════════════════════════════════════════════

np.random.seed(123)

# Simular ventas diarias de una tienda (30 días)
n_dias = 30

# Ventas base con variación normal
ventas_base = 1000
variacion_std = 150
ventas_simuladas = np.random.normal(ventas_base, variacion_std, n_dias)

# Agregar efecto de fin de semana (días 6, 7, 13, 14, etc.)
dias = np.arange(1, n_dias + 1)
es_finde = (dias % 7 == 6) | (dias % 7 == 0)
ventas_simuladas[es_finde] *= 1.3  # 30% más en fines de semana

# Asegurar valores positivos
ventas_simuladas = np.maximum(ventas_simuladas, 0)

print("🛒 SIMULACIÓN DE VENTAS DIARIAS")
print("═" * 60)
print(f"\nParámetros:")
print(f"   Ventas base: ${ventas_base}")
print(f"   Variación (std): ${variacion_std}")
print(f"   Incremento fin de semana: +30%")

print(f"\n📊 ESTADÍSTICAS DE LA SIMULACIÓN:")
print(f"   Media:    ${np.mean(ventas_simuladas):,.0f}")
print(f"   Mediana:  ${np.median(ventas_simuladas):,.0f}")
print(f"   Mínimo:   ${np.min(ventas_simuladas):,.0f}")
print(f"   Máximo:   ${np.max(ventas_simuladas):,.0f}")
print(f"   Std Dev:  ${np.std(ventas_simuladas):,.0f}")

print(f"\n📅 Muestra de ventas (primeros 7 días):")
for i in range(7):
    tipo = "(fin de semana)" if es_finde[i] else ""
    print(f"   Día {i+1}: ${ventas_simuladas[i]:,.0f} {tipo}")

---

## 8. Resumen y Funciones Útiles

### Tabla de Funciones Estadísticas de NumPy

```
FUNCIONES ESTADÍSTICAS DE NUMPY
═══════════════════════════════════════════════════════════════════════════════

TENDENCIA CENTRAL:
─────────────────────────────────────────────────────────────────────────────
  np.mean(x)           Media aritmética
  np.median(x)         Mediana (valor central)
  np.average(x, w)     Media ponderada

DISPERSIÓN:
─────────────────────────────────────────────────────────────────────────────
  np.std(x)            Desviación estándar
  np.var(x)            Varianza
  np.ptp(x)            Rango (peak-to-peak: max - min)

POSICIÓN:
─────────────────────────────────────────────────────────────────────────────
  np.percentile(x, p)  Percentil p de x
  np.quantile(x, q)    Cuantil q de x (q entre 0 y 1)
  np.min(x), np.max(x) Mínimo y máximo

RELACIÓN:
─────────────────────────────────────────────────────────────────────────────
  np.corrcoef(x, y)    Matriz de correlación
  np.cov(x, y)         Matriz de covarianza

CON VALORES NaN:
─────────────────────────────────────────────────────────────────────────────
  np.nanmean(x)        Media ignorando NaN
  np.nanstd(x)         Std ignorando NaN
  np.nanpercentile(x)  Percentil ignorando NaN

═══════════════════════════════════════════════════════════════════════════════
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                FUNCIÓN DE ANÁLISIS ESTADÍSTICO COMPLETO
# ═══════════════════════════════════════════════════════════════════

def analisis_estadistico(datos, nombre="Variable"):
    """
    Realiza un análisis estadístico completo de un array.
    
    Parámetros:
    -----------
    datos : array-like
        Datos a analizar
    nombre : str
        Nombre de la variable para el reporte
    
    Retorna:
    --------
    dict con todas las estadísticas calculadas
    """
    datos = np.asarray(datos)
    
    # Estadísticas básicas
    n = len(datos)
    n_validos = np.sum(~np.isnan(datos))
    n_nulos = np.sum(np.isnan(datos))
    
    # Tendencia central
    media = np.nanmean(datos)
    mediana = np.nanmedian(datos)
    
    # Dispersión
    std = np.nanstd(datos)
    var = np.nanvar(datos)
    minimo = np.nanmin(datos)
    maximo = np.nanmax(datos)
    rango = maximo - minimo
    
    # Cuartiles e IQR
    Q1 = np.nanpercentile(datos, 25)
    Q3 = np.nanpercentile(datos, 75)
    IQR = Q3 - Q1
    
    # Outliers
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    n_outliers = np.sum((datos < lim_inf) | (datos > lim_sup))
    
    # Forma
    datos_validos = datos[~np.isnan(datos)]
    skewness = calcular_skewness(datos_validos)
    kurtosis = calcular_kurtosis(datos_validos)
    
    # Imprimir reporte
    print(f"\n📊 ANÁLISIS ESTADÍSTICO: {nombre}")
    print("═" * 55)
    print(f"\n📈 MUESTRA:")
    print(f"   Total observaciones:  {n}")
    print(f"   Valores válidos:      {n_validos}")
    print(f"   Valores nulos:        {n_nulos}")
    
    print(f"\n📍 TENDENCIA CENTRAL:")
    print(f"   Media:                {media:.4f}")
    print(f"   Mediana:              {mediana:.4f}")
    
    print(f"\n📐 DISPERSIÓN:")
    print(f"   Desv. Estándar:       {std:.4f}")
    print(f"   Varianza:             {var:.4f}")
    print(f"   Rango:                {rango:.4f}")
    print(f"   Mínimo:               {minimo:.4f}")
    print(f"   Máximo:               {maximo:.4f}")
    
    print(f"\n📊 CUARTILES:")
    print(f"   Q1 (25%):             {Q1:.4f}")
    print(f"   Q2 (50%):             {mediana:.4f}")
    print(f"   Q3 (75%):             {Q3:.4f}")
    print(f"   IQR:                  {IQR:.4f}")
    
    print(f"\n⚠️  OUTLIERS (método IQR):")
    print(f"   Límite inferior:      {lim_inf:.4f}")
    print(f"   Límite superior:      {lim_sup:.4f}")
    print(f"   Cantidad detectada:   {n_outliers}")
    
    print(f"\n📐 FORMA:")
    print(f"   Asimetría (skewness): {skewness:.4f}")
    print(f"   Curtosis:             {kurtosis:.4f}")
    
    return {
        'n': n, 'n_validos': n_validos, 'n_nulos': n_nulos,
        'media': media, 'mediana': mediana,
        'std': std, 'var': var, 'min': minimo, 'max': maximo, 'rango': rango,
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'lim_inf': lim_inf, 'lim_sup': lim_sup, 'n_outliers': n_outliers,
        'skewness': skewness, 'kurtosis': kurtosis
    }

# Ejemplo de uso
np.random.seed(42)
datos_ejemplo = np.concatenate([
    np.random.normal(100, 15, 95),
    [5, 200, np.nan, np.nan, 250]  # Outliers y nulos
])

resultado = analisis_estadistico(datos_ejemplo, "Puntajes de Prueba")

---

## 9. Ejercicios de Práctica

### Ejercicio 1: Análisis de Temperaturas
Dado un array de temperaturas diarias, calcula los cuartiles y detecta días con temperatura anormal.

In [ ]:
# EJERCICIO 1: Completa el código

# Temperaturas diarias (°C) de un mes
temperaturas = np.array([22, 24, 21, 23, 25, 26, 28, 30, 32, 29,
                         27, 25, 23, 22, 20, 19, 18, 17, 19, 21,
                         23, 25, 27, 29, 31, 5, 33, 35, 34, 45])

# TODO: Calcula Q1, Q3, IQR
Q1 = ___
Q3 = ___
IQR = ___

# TODO: Calcula los límites para outliers
limite_inferior = ___
limite_superior = ___

# TODO: Identifica los días con temperatura anormal
dias_anormales = ___

print(f"Q1: {Q1}°C, Q3: {Q3}°C, IQR: {IQR}°C")
print(f"Límites: [{limite_inferior}°C, {limite_superior}°C]")
print(f"Días con temperatura anormal: {dias_anormales}")

### Ejercicio 2: Correlación de Variables

In [ ]:
# EJERCICIO 2: Completa el código

# Datos de publicidad y ventas
np.random.seed(42)
inversion_publicidad = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
ventas = 100 + 2.5 * inversion_publicidad + np.random.normal(0, 10, 10)

# TODO: Calcula la correlación entre inversión y ventas
correlacion = ___

# TODO: Interpreta la fuerza de la correlación
if abs(correlacion) > 0.7:
    fuerza = "___"
elif abs(correlacion) > 0.3:
    fuerza = "___"
else:
    fuerza = "___"

print(f"Correlación: {correlacion:.4f}")
print(f"Interpretación: Correlación {fuerza}")

---

## Resumen de la Semana 9

```
╔══════════════════════════════════════════════════════════════════════════╗
║                    RESUMEN: NUMPY Y ESTADÍSTICA                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  📊 PERCENTILES Y CUARTILES                                              ║
║     • np.percentile(datos, p)  - Percentil p                             ║
║     • np.quantile(datos, q)    - Cuantil q (0-1)                         ║
║     • Q1, Q2 (mediana), Q3 dividen datos en 4 partes                     ║
║                                                                          ║
║  🔍 DETECCIÓN DE OUTLIERS                                                ║
║     • Método IQR: valores fuera de [Q1-1.5*IQR, Q3+1.5*IQR]              ║
║     • Método Z-Score: |z| > 3 indica outlier                             ║
║                                                                          ║
║  📐 DISTRIBUCIONES                                                       ║
║     • np.histogram() - Frecuencias por intervalo                         ║
║     • Skewness - Mide asimetría (0 = simétrica)                          ║
║     • Kurtosis - Mide colas (0 = normal)                                 ║
║                                                                          ║
║  📈 CORRELACIÓN                                                          ║
║     • np.corrcoef() - Matriz de correlación (-1 a 1)                     ║
║     • np.cov() - Matriz de covarianza                                    ║
║     • |r| > 0.7 = fuerte, 0.3-0.7 = moderada, <0.3 = débil               ║
║                                                                          ║
║  🔢 OPERACIONES MATRICIALES                                              ║
║     • A.T - Transposición                                                ║
║     • np.dot(A, B) o A @ B - Producto punto                              ║
║     • axis=0 (filas), axis=1 (columnas)                                  ║
║                                                                          ║
║  🎲 GENERACIÓN ALEATORIA                                                 ║
║     • np.random.seed() - Reproducibilidad                                ║
║     • np.random.normal(), uniform(), randint(), choice()                 ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
```

---

### Próxima Semana
**Semana 10: Pandas Fundamentos** - Series, DataFrames, carga y limpieza de datos.

---

*Material desarrollado para Programación para Ciencia de Datos - IPN*